In [1]:
import pandas as pd

In [36]:
# Read tables
dataset = pd.read_csv("../output/manual_labeled_dataset.csv").T.reset_index()
# Change column names
dataset.columns = dataset.iloc[0]
# delete first row
dataset = dataset.iloc[1:]
# remove the added redundant index
dataset.doc_id = dataset.doc_id.apply(lambda x: x.split(".")[0])
# drop rows with all nan values
dataset = dataset.dropna(subset=dataset.columns[2:], how='all')

In [47]:
pred_df = pd.read_csv("../output/gpt55/predict.csv", index_col=0)

In [48]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score

def evaluate_sdoh_datasets(df_true, df_pred):
    # 1. Define Keys and Features
    key_cols = ['doc_id', 'Experiencer']
    feature_cols = [col for col in df_true.columns if col not in key_cols]
    
    # 2. Preprocessing: Standardize text and handle missing values
    # We replace NaN with 'not_mentioned' so the metrics can evaluate empty cells properly
    fill_value = 'not_mentioned'
    
    df_true_clean = df_true.fillna(fill_value).astype(str).apply(lambda x: x.str.lower().str.strip())
    df_pred_clean = df_pred.fillna(fill_value).astype(str).apply(lambda x: x.str.lower().str.strip())

    # 3. Align datasets using an Outer Join
    merged = pd.merge(df_true_clean, df_pred_clean, on=key_cols, how='outer', indicator=True, suffixes=('_true', '_pred'))
    
    # ==========================================
    # STAGE A: ROW-LEVEL METRICS (Identification)
    # ==========================================
    tp_rows = (merged['_merge'] == 'both').sum()
    fp_rows = (merged['_merge'] == 'right_only').sum()
    fn_rows = (merged['_merge'] == 'left_only').sum()
    
    row_precision = tp_rows / (tp_rows + fp_rows) if (tp_rows + fp_rows) > 0 else 0.0
    row_recall = tp_rows / (tp_rows + fn_rows) if (tp_rows + fn_rows) > 0 else 0.0
    row_f1 = 2 * (row_precision * row_recall) / (row_precision + row_recall) if (row_precision + row_recall) > 0 else 0.0
    
    print("=== STAGE A: ROW-LEVEL IDENTIFICATION (doc_id + Experiencer) ===")
    print(f"Matched Rows (TP): {tp_rows}")
    print(f"Hallucinated Rows (FP): {fp_rows}")
    print(f"Missed Rows (FN): {fn_rows}")
    print(f"Row Precision: {row_precision:.4f} | Row Recall: {row_recall:.4f} | Row F1: {row_f1:.4f}\n")

    # ==========================================
    # STAGE B: COLUMN-LEVEL METRICS (Attribute Accuracy)
    # ==========================================
    # Filter to only the rows where both datasets agreed on the experiencer
    matched_df = merged[merged['_merge'] == 'both'].copy()
    
    if tp_rows == 0:
        print("No matching rows found. Cannot compute Stage B metrics.")
        return None

    # Calculate Exact Match Ratio (Rows where ALL features match perfectly)
    exact_matches = 0
    for _, row in matched_df.iterrows():
        is_exact = all(row[f"{col}_true"] == row[f"{col}_pred"] for col in feature_cols)
        if is_exact:
            exact_matches += 1
            
    emr = exact_matches / tp_rows
    
    print("=== STAGE B: OVERALL COLUMN METRICS ===")
    print(f"Exact Match Ratio (Perfect Rows): {emr:.4f}")

    # Calculate Micro-F1 (treating every single cell in the table as an individual prediction)
    all_true_cells = []
    all_pred_cells = []
    
    for col in feature_cols:
        all_true_cells.extend(matched_df[f"{col}_true"].tolist())
        all_pred_cells.extend(matched_df[f"{col}_pred"].tolist())
        
    micro_f1 = f1_score(all_true_cells, all_pred_cells, average='micro', zero_division=0)
    print(f"Global Micro-F1 Score: {micro_f1:.4f}\n")

    # Calculate Per-Column Metrics (Cohen's Kappa and Macro-F1)
    col_results = []
    
    for col in feature_cols:
        y_true = matched_df[f"{col}_true"]
        y_pred = matched_df[f"{col}_pred"]
        
        # Calculate Accuracy
        acc = accuracy_score(y_true, y_pred)
        
        # Calculate Macro F1
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
        # Calculate Cohen's Kappa
        # Warning: If all true and pred values are identical (e.g., all 'not_mentioned'), 
        # Kappa can throw a warning or return NaN. We catch this by checking unique values.
        unique_labels = set(y_true).union(set(y_pred))
        if len(unique_labels) > 1:
            kappa = cohen_kappa_score(y_true, y_pred)
        else:
            kappa = 1.0 if acc == 1.0 else 0.0 # Perfect agreement on a single label
            
        col_results.append({
            'Feature': col,
            'Accuracy': acc,
            'Cohen_Kappa': kappa,
            'Macro_F1': macro_f1
        })
        
    # Convert results to a DataFrame for easy viewing/exporting
    results_df = pd.DataFrame(col_results)
    
    print("=== PER-COLUMN BREAKDOWN ===")
    # Print the top 5 worst performing columns so you know where the LLM struggles
    worst_cols = results_df.sort_values(by='Cohen_Kappa').head(5)
    print("Top 5 Hardest Features for the LLM (Lowest Kappa):")
    print(worst_cols.to_string(index=False))
    
    return results_df


In [49]:

detailed_metrics = evaluate_sdoh_datasets(dataset, pred_df)


=== STAGE A: ROW-LEVEL IDENTIFICATION (doc_id + Experiencer) ===
Matched Rows (TP): 113
Hallucinated Rows (FP): 122
Missed Rows (FN): 41
Row Precision: 0.4809 | Row Recall: 0.7338 | Row F1: 0.5810

=== STAGE B: OVERALL COLUMN METRICS ===
Exact Match Ratio (Perfect Rows): 0.0885
Global Micro-F1 Score: 0.7895

=== PER-COLUMN BREAKDOWN ===
Top 5 Hardest Features for the LLM (Lowest Kappa):
               Feature  Accuracy  Cohen_Kappa  Macro_F1
 trauma_physical_abuse  0.716814    -0.016873  0.278351
           mental_ptsd  0.973451    -0.005935  0.328849
  number of caregivers  0.964602     0.000000  0.490991
mental_neuro_condition  0.973451     0.000000  0.328849
         trauma_arrest  0.876106     0.000000  0.233491


In [51]:
detailed_metrics

,Feature,Accuracy,Cohen_Kappa,Macro_F1
0,financial_status,0.504425,0.114594,0.281067
1,employment_status,0.761062,0.658801,0.748585
2,education_status,0.858407,0.693715,0.727173
3,education_level,0.743363,0.497470,0.417482
4,healthcare_type,0.805310,0.385870,0.346834
5,social_family_level,0.619469,0.306551,0.383115
6,social_church_level,0.690265,0.219305,0.397113
7,social_nonprofit_level,0.982301,0.000000,0.330357
8,social_government_level,0.654867,0.032704,0.210455
9,mental_adhd,0.982301,0.000000,0.495536
